# 01 — Data Understanding

**Owner:** Yasasri's lane — structure, dtypes, missingness, duplicates, target balance.

Remember: every important choice gets a decision-log cell below it (what we decided, evidence, alternative considered, why rejected). These become your viva script and report paragraphs — see `CLAUDE.md`.

In [ ]:
import sys
from pathlib import Path

# Jupyter's kernel CWD is notebooks/, so add the repo root to sys.path
# before importing anything from src/.
sys.path.append(str(Path.cwd().parent))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RANDOM_STATE, FIGURES_DIR
from src.pipeline import load_data

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
df = load_data()
df.shape

In [ ]:
df.head()

## Column role table

Every column is classified by its role in the modelling pipeline, its raw dtype after loading, and a one-line description.
This table feeds directly into report section 4 (Data Understanding).

In [ ]:
# Column metadata: role, dtype, description
# Roles: identifier | timestamp | sensor_feature | target | event_flag
column_meta = [
    ("Num",                   "identifier",     "int64",              "Sequential row index from the original log file"),
    ("Timestamp",             "timestamp",      "datetime64[ns,UTC]", "UTC wall-clock time of the reading (~1 s cadence)"),
    ("cycle",                 "identifier",     "int64",              "Cycle number grouping rows into discrete robot work cycles"),
    ("Current_J0",            "sensor_feature", "float64",            "Motor current (A) for joint 0 (base rotation)"),
    ("Current_J1",            "sensor_feature", "float64",            "Motor current (A) for joint 1 (shoulder)"),
    ("Current_J2",            "sensor_feature", "float64",            "Motor current (A) for joint 2 (elbow)"),
    ("Current_J3",            "sensor_feature", "float64",            "Motor current (A) for joint 3 (wrist 1)"),
    ("Current_J4",            "sensor_feature", "float64",            "Motor current (A) for joint 4 (wrist 2)"),
    ("Current_J5",            "sensor_feature", "float64",            "Motor current (A) for joint 5 (wrist 3)"),
    ("Temperature_T0",        "sensor_feature", "float64",            "Temperature (°C) at joint 0"),
    ("Temperature_J1",        "sensor_feature", "float64",            "Temperature (°C) at joint 1"),
    ("Temperature_J2",        "sensor_feature", "float64",            "Temperature (°C) at joint 2"),
    ("Temperature_J3",        "sensor_feature", "float64",            "Temperature (°C) at joint 3"),
    ("Temperature_J4",        "sensor_feature", "float64",            "Temperature (°C) at joint 4"),
    ("Temperature_J5",        "sensor_feature", "float64",            "Temperature (°C) at joint 5"),
    ("Speed_J0",              "sensor_feature", "float64",            "Joint angular speed (rad/s) for joint 0"),
    ("Speed_J1",              "sensor_feature", "float64",            "Joint angular speed (rad/s) for joint 1"),
    ("Speed_J2",              "sensor_feature", "float64",            "Joint angular speed (rad/s) for joint 2"),
    ("Speed_J3",              "sensor_feature", "float64",            "Joint angular speed (rad/s) for joint 3"),
    ("Speed_J4",              "sensor_feature", "float64",            "Joint angular speed (rad/s) for joint 4"),
    ("Speed_J5",              "sensor_feature", "float64",            "Joint angular speed (rad/s) for joint 5"),
    ("Tool_current",          "sensor_feature", "float64",            "Current (A) drawn by the end-effector tool"),
    ("Robot_ProtectiveStop",  "target",         "float64→int",        "1 = protective stop triggered, 0 = normal operation (the label to predict)"),
    ("grip_lost",             "event_flag",     "bool",               "True = gripper lost its object during this reading (correlated event, not a stop)"),
]

role_df = pd.DataFrame(column_meta, columns=["Column", "Role", "Dtype", "Description"])
role_df

## Dtype fixes

`Robot_ProtectiveStop` loaded as `float64` because the 54 missing rows forced pandas to use NaN (which requires float).
After we decide what to do with those rows (see Missingness section below), we cast it to a nullable integer (`Int8`) so it is
unambiguous as a binary label and consistent with the `bool` type of `grip_lost`.

In [ ]:
# Before fix
print("Before:", df["Robot_ProtectiveStop"].dtype)

# Cast to nullable Int8 (preserves NaN rows before we drop them)
df["Robot_ProtectiveStop"] = df["Robot_ProtectiveStop"].astype("Int8")

print("After: ", df["Robot_ProtectiveStop"].dtype)
df[["Robot_ProtectiveStop", "grip_lost"]].dtypes

## Missingness

The project plan states: *54 rows have missing values, all 54 are also missing the target, 46 missing every sensor.*
We confirm this finding and decide what to do with those rows.

In [ ]:
# Missing value count per column
miss = df.isna().sum().rename("n_missing")
miss_pct = (miss / len(df) * 100).rename("pct_missing").round(2)
pd.concat([miss, miss_pct], axis=1)[miss > 0]

In [ ]:
# Confirm the plan's three claims
missing_rows = df[df.isna().any(axis=1)]
n_missing_rows = len(missing_rows)

target_also_missing = missing_rows["Robot_ProtectiveStop"].isna().sum()

sensor_cols = [c for c in df.columns
               if c not in ("Num", "Timestamp", "cycle", "Robot_ProtectiveStop", "grip_lost")]
# 'all sensors missing' means ALL 21 sensor columns are NaN
all_sensors_missing = missing_rows[sensor_cols].isna().all(axis=1).sum()

print(f"Total rows with any NaN:           {n_missing_rows}  (plan says 54)")
print(f"Of those, target also missing:     {target_also_missing}  (plan says 54)")
print(f"Of those, ALL 21 sensors missing:  {all_sensors_missing}  (plan says 46)")
print()
# Note: Current_J0 has only 46 missing, not 54 — so 8 rows have some sensors but no target
partial_sensor_rows = n_missing_rows - all_sensors_missing
print(f"Rows with PARTIAL sensors missing: {partial_sensor_rows}  (these are the 8 remaining rows)")

In [ ]:
# Missingness heatmap — show only rows that have at least one NaN
fig, ax = plt.subplots(figsize=(14, 4))

# Build a boolean mask (True = missing)
miss_matrix = missing_rows[["Robot_ProtectiveStop"] + sensor_cols].isna()

sns.heatmap(
    miss_matrix.T,
    cbar=False,
    cmap="Reds",
    linewidths=0.3,
    ax=ax,
    yticklabels=True,
)
ax.set_title(
    f"Missingness pattern across the {n_missing_rows} affected rows\n"
    f"(Red = missing | each column is a sensor or the target label)",
    fontsize=11,
)
ax.set_xlabel("Row index within the 54 missing rows")
ax.set_ylabel("Column")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig01_missingness.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/fig01_missingness.png")

### Decision log — handling the 54 missing rows

**What we decided:** Drop all 54 rows with missing values.

**Evidence:**
- All 54 rows have a missing target (`Robot_ProtectiveStop = NaN`), so they cannot be used for supervised learning under any strategy.
- 46 of the 54 are missing all 21 sensor columns as well — there is no feature information to impute from.
- The remaining 8 rows have partial sensors but still no label; they are unusable as training examples.
- 54 rows is 0.73% of 7,409 — a negligible share of the dataset.

**Alternative considered:** Impute the sensor values for those 8 partially-missing rows and keep them as unlabelled data for semi-supervised learning.

**Why rejected:** Semi-supervised learning adds significant complexity and is out of scope for this project. The 8 rows represent 0.1% of the data; any information gain is immaterial. Dropping them is the simplest, most defensible choice and avoids introducing imputed values near the decision boundary.

In [ ]:
# Apply the decision: drop all rows where the target is missing
df_clean = df.dropna(subset=["Robot_ProtectiveStop"]).copy()
df_clean["Robot_ProtectiveStop"] = df_clean["Robot_ProtectiveStop"].astype("int8")

print(f"Rows before drop: {len(df):,}")
print(f"Rows after drop:  {len(df_clean):,}  (dropped {len(df) - len(df_clean)} rows)")
print(f"Remaining NaNs:   {df_clean.isna().sum().sum()}")

## Duplicate check

We check for two kinds of duplicates:
1. **Exact row duplicates** — all 24 column values identical.
2. **Duplicate timestamps** — same UTC second appearing more than once (would break temporal ordering in the split).

In [ ]:
exact_dups = df_clean.duplicated().sum()
ts_dups    = df_clean.duplicated(subset=["Timestamp"]).sum()

print(f"Exact duplicate rows:      {exact_dups}")
print(f"Duplicate Timestamps:      {ts_dups}")

if exact_dups == 0 and ts_dups == 0:
    print("\n✓ No duplicates found — the dataset is temporally unique row-by-row.")

### Decision log — duplicates

**What we decided:** No action required.

**Evidence:** Zero exact duplicate rows and zero duplicate timestamps in the cleaned dataset (7,355 rows). Every row has a unique UTC timestamp at approximately 1-second cadence.

**Alternative considered:** Checking for near-duplicate rows (e.g., same timestamp ± 1 s, very similar sensor values).

**Why rejected:** The dataset logs from a single robot at ~1 Hz; sensor values naturally vary between consecutive seconds. Near-duplicate detection would require an arbitrary similarity threshold and is not warranted when exact duplicates are already zero.

## Target balance

We confirm the expected class distribution for `Robot_ProtectiveStop` and explain why it means accuracy is a misleading metric.

In [ ]:
vc = df_clean["Robot_ProtectiveStop"].value_counts().sort_index()
total = len(df_clean)
pos = int(vc.get(1, 0))
neg = int(vc.get(0, 0))
pos_rate = pos / total * 100

print(f"Class 0 (normal):          {neg:,}  ({neg/total*100:.2f}%)")
print(f"Class 1 (protective stop): {pos:,}   ({pos_rate:.2f}%)")
print(f"Total (after dropping NaN rows): {total:,}")
print()
print(f"Imbalance ratio (majority:minority):  {neg/pos:.1f}:1")
print()
dummy_accuracy = neg / total * 100
print(f"A dummy classifier that always predicts 0 would score {dummy_accuracy:.2f}% accuracy.")
print("This is why we use PR-AUC as the primary metric, not accuracy.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# --- Left: bar chart ---
ax = axes[0]
bars = ax.bar(
    ["Normal (0)", "Protective Stop (1)"],
    [neg, pos],
    color=["#4C9BE8", "#E85C4C"],
    edgecolor="white",
    width=0.5,
)
for bar, n in zip(bars, [neg, pos]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{n:,}\n({n/total*100:.1f}%)",
        ha="center", va="bottom", fontsize=10,
    )
ax.set_title("Class distribution — Robot_ProtectiveStop", fontsize=11)
ax.set_ylabel("Row count")
ax.set_ylim(0, neg * 1.15)

# --- Right: pie chart ---
ax2 = axes[1]
ax2.pie(
    [neg, pos],
    labels=[f"Normal\n{neg:,} ({neg/total*100:.1f}%)",
            f"Stop\n{pos} ({pos_rate:.2f}%)"],
    colors=["#4C9BE8", "#E85C4C"],
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    autopct=None,
)
ax2.set_title("Class share (pie)", fontsize=11)

plt.suptitle(
    f"Target imbalance: {pos_rate:.2f}% positive  |  {neg/pos:.1f}:1 majority-to-minority ratio",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig02_target_balance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/fig02_target_balance.png")

### Decision log — primary metric choice

**What we decided:** Use **PR-AUC (average precision)** as the primary evaluation metric throughout the project.

**Evidence:**
- Only 278 of 7,355 rows (3.78%) are positive. The majority-to-minority ratio is 26.5:1.
- A dummy classifier that always predicts "normal" achieves 96.22% accuracy — yet it catches zero stops.
- PR-AUC summarises precision–recall trade-offs across all thresholds, giving full weight to performance on the minority (positive) class. It is the standard metric for highly imbalanced binary classification.

**Alternative considered:** ROC-AUC as the sole metric.

**Why rejected:** ROC-AUC can be optimistically high for imbalanced datasets because it also accounts for true negatives, which are trivially easy to get right when negatives dominate. PR-AUC is a strictly harder and more informative criterion when false negatives (missed stops) are costly — as they are in a safety-critical robotics setting.

## Summary — Decision log

| # | Decision | Evidence | Alternative | Why rejected |
|---|----------|----------|-------------|---------------|
| 1 | Drop all 54 rows with missing target | All 54 have `Robot_ProtectiveStop = NaN`; 46 also have every sensor missing; only 0.73% of data | Impute the 8 partially-missing rows and use as unlabelled data | Semi-supervised out of scope; 0.1% information gain is immaterial |
| 2 | No duplicate removal needed | 0 exact duplicate rows, 0 duplicate timestamps in 7,355 rows | Near-duplicate detection with similarity threshold | No exact duplicates found; arbitrary threshold not warranted |
| 3 | Cast `Robot_ProtectiveStop` from float64 to Int8 | Column is binary (0/1); float was forced by NaN rows before dropping | Keep as float64 | Int8 is semantically correct for a binary label and saves memory |
| 4 | Use PR-AUC as primary metric | 3.78% positive rate; dummy-always-0 gets 96.22% accuracy yet catches no stops | ROC-AUC as sole metric | ROC-AUC inflated by easy true negatives in imbalanced settings; PR-AUC is harder and directly measures minority-class performance |